# Phase 1 — Week 1: Setup & Foundation

**Goal:** Build the foundation of our emotional audiobook system.

By the end of this notebook you will have:
1. ✅ A working Python environment
2. ✅ Downloaded 3 public-domain test stories
3. ✅ Loaded one story cleanly (no Gutenberg boilerplate)
4. ✅ Split it into sentences and paragraphs
5. ✅ A clean foundation to build emotion analysis on next week

**Reminder:** Run cells in order (Shift+Enter).

## Step 1 — Set up imports

We add the project's `src` folder to Python's path so we can import our own modules.

In [1]:
import sys
from pathlib import Path

# Add the project root to sys.path so we can `import src`.
# This notebook lives in <project>/notebooks, so go up one level.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python: {sys.version.split()[0]}")

Project root: /Users/kavindumihiranga/Documents/NextJS/emotional-audiobook/V3/emotional_audiobook
Python: 3.11.15


In [2]:
from src import story_downloader, text_loader, segmenter

print("✅ All Phase 1 modules imported successfully.")

✅ All Phase 1 modules imported successfully.


## Step 2 — Download test stories

We fetch 3 short public-domain stories from Project Gutenberg.
Each has a clear emotional arc, which makes them good test cases for next week.

*If you have no internet right now, skip this and use your own .txt file in `data/raw/`.*

In [3]:
raw_data_dir = project_root / "data" / "raw"
downloaded = story_downloader.download_all(raw_data_dir)

print("\nAvailable stories:")
for key, path in downloaded.items():
    print(f"  • {key:25s} -> {path.name}")

Stories:   0%|                                                                                                             | 0/3 [00:00<?, ?it/s]

Downloading: The Gift of the Magi by O. Henry


Stories:  33%|█████████████████████████████████▋                                                                   | 1/3 [00:36<01:12, 36.19s/it]

  saved -> /Users/kavindumihiranga/Documents/NextJS/emotional-audiobook/V3/emotional_audiobook/data/raw/gift_of_the_magi.txt (31,144 chars)
Downloading: The Necklace by Guy de Maupassant


Stories:  67%|███████████████████████████████████████████████████████████████████▎                                 | 2/3 [00:58<00:28, 28.07s/it]

  saved -> /Users/kavindumihiranga/Documents/NextJS/emotional-audiobook/V3/emotional_audiobook/data/raw/the_necklace.txt (2,771,322 chars)
Downloading: The Tell-Tale Heart by Edgar Allan Poe


Stories: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:06<00:00, 22.09s/it]

  saved -> /Users/kavindumihiranga/Documents/NextJS/emotional-audiobook/V3/emotional_audiobook/data/raw/tell_tale_heart.txt (602,212 chars)

Available stories:
  • gift_of_the_magi          -> gift_of_the_magi.txt
  • the_necklace              -> the_necklace.txt
  • tell_tale_heart           -> tell_tale_heart.txt


## Step 3 — Load and clean one story

Let's pick one and run it through our text loader.
The loader strips Gutenberg's license header/footer and normalizes whitespace.

In [4]:
# Pick which story to work with. Change this to try the others.
STORY_KEY = "gift_of_the_magi"

# load_story_by_key applies the section markers for anthology files.
story_text = story_downloader.load_story_by_key(STORY_KEY, raw_data_dir)

stats = text_loader.get_basic_stats(story_text)
print(f"Loaded: {STORY_KEY}")
print(f"  Characters:  {stats['characters']:,}")
print(f"  Words:       {stats['words']:,}")
print(f"  Paragraphs:  {stats['paragraphs']:,}")

print("\nFirst 500 characters of the cleaned story:")
print("-" * 60)
print(story_text[:500])
print("-" * 60)

Loaded: gift_of_the_magi
  Characters:  11,201
  Words:       2,065
  Paragraphs:  47

First 500 characters of the cleaned story:
------------------------------------------------------------
The Gift of the Magi

by O. Henry

One dollar and eighty-seven cents. That was all. And sixty cents of it
was in pennies. Pennies saved one and two at a time by bulldozing the
grocer and the vegetable man and the butcher until one’s cheeks burned
with the silent imputation of parsimony that such close dealing
implied. Three times Della counted it. One dollar and eighty-seven
cents. And the next day would be Christmas.

There was clearly nothing to do but flop down on the shabby little
couch and 
------------------------------------------------------------


## Step 4 — Segment the story

Now we split the cleaned text into:
- **Sentences** — fine-grained, we'll use these for emotion labels in Week 2
- **Paragraphs** — coarser, useful as context for summarization in Week 3

In [5]:
segments = segmenter.segment_story(story_text)

print(f"Segmentation results for '{STORY_KEY}':")
for key, val in segments["stats"].items():
    if isinstance(val, float):
        print(f"  {key:35s} {val:.1f}")
    else:
        print(f"  {key:35s} {val:,}")

Segmentation results for 'gift_of_the_magi':
  num_sentences                       141
  num_paragraphs                      47
  avg_sentence_length_words           14.6
  avg_paragraph_length_words          43.9


In [6]:
# Spot-check: print the first 5 sentences. This is also a quick QA
# step — if these don't look like real sentence boundaries, something
# went wrong in segmentation.
print("First 5 sentences:\n")
for i, sent in enumerate(segments["sentences"][:5], 1):
    print(f"[{i}] {sent}\n")

First 5 sentences:

[1] The Gift of the Magi by O. Henry One dollar and eighty-seven cents.

[2] That was all.

[3] And sixty cents of it was in pennies.

[4] Pennies saved one and two at a time by bulldozing the grocer and the vegetable man and the butcher until one’s cheeks burned with the silent imputation of parsimony that such close dealing implied.

[5] Three times Della counted it.



In [7]:
# And the first 2 paragraphs — these are what Week 3's summarizer
# will consume.
print("First 2 paragraphs:\n")
for i, para in enumerate(segments["paragraphs"][:2], 1):
    print(f"[Paragraph {i}]")
    print(para)
    print()

First 2 paragraphs:

[Paragraph 1]
The Gift of the Magi

[Paragraph 2]
by O. Henry



## Step 5 — Save the processed output

We save the segmented story to disk as JSON. Next week's emotion-analysis
notebook will load this directly instead of re-running segmentation.

In [8]:
import json

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / f"{STORY_KEY}_segmented.json"
with output_path.open("w", encoding="utf-8") as f:
    json.dump(segments, f, ensure_ascii=False, indent=2)

print(f"✅ Saved -> {output_path}")
print(f"   File size: {output_path.stat().st_size:,} bytes")

✅ Saved -> /Users/kavindumihiranga/Documents/NextJS/emotional-audiobook/V3/emotional_audiobook/data/processed/gift_of_the_magi_segmented.json
   File size: 24,424 bytes


## Step 6 — Process the remaining stories

Same pipeline, but applied to every story we downloaded. This builds up the dataset Week 2 will work on.

In [9]:
for key in downloaded:
    text = story_downloader.load_story_by_key(key, raw_data_dir)
    segs = segmenter.segment_story(text)

    out_path = processed_dir / f"{key}_segmented.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(segs, f, ensure_ascii=False, indent=2)

    print(
        f"  ✅ {key:25s} "
        f"{segs['stats']['num_sentences']:4d} sentences, "
        f"{segs['stats']['num_paragraphs']:3d} paragraphs"
    )

print("\n✅ All stories processed and saved.")

  ✅ gift_of_the_magi           141 sentences,  47 paragraphs
  ✅ the_necklace              3153 sentences, 1812 paragraphs
  ✅ tell_tale_heart            154 sentences,  19 paragraphs

✅ All stories processed and saved.


## Week 1 — Complete ✅

**What we built:**
- A clean, modular project structure
- A reusable story downloader (Project Gutenberg)
- A text loader that handles real-world messy book text
- A segmenter that produces sentence- and paragraph-level chunks

**What's next (Week 2 — Emotion Analysis):**
- Load the segmented JSON files we just produced
- Run each sentence through a pre-trained emotion classifier
- Tag every sentence with an emotion + confidence score
- Plot the story's "emotional arc"